# ЛР 04 — ноутбук 1 (todo): основы калибровки вероятностей

## Как работать с этим ноутбуком
- Это версия с пошаговым заполнением для новичков: основа уже подготовлена, ключевые места отмечены `TODO(обязательно)`.
- На каждом шаге заполняйте блок «Мини-вывод» своими словами.
- В этом ноутбуке используем только `validation` (без финальной проверки на `test`).
- В конце оставлена намеренная остановка: экспорт включаете только после самостоятельной проверки.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

for candidate in [Path.cwd(), Path.cwd().parent]:
    if str(candidate) not in sys.path:
        sys.path.append(str(candidate))

LAB_DIR = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent] if (candidate / 'lab_utils.py').exists()),
    Path.cwd(),
)
OUTPUT_DIR = LAB_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import lab_utils as lab

np.random.seed(lab.SEED)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


## Шаг 1. Подготовка контекста и входных гипотез

### Что делаем
Загружаем наборы данных, наборы признаков и гипотезы из ЛР 03.

### Зачем
Чтобы дальше честно сравнивать варианты калибровки на одинаковом входе.

### Вход
- `baseline_vs_tuned_test_results.csv` из ЛР 03
- `feature_sets_wrapper_embedded.json` из ЛР 01

### Выход
Таблица `hypotheses` по двум наборам данных: `medical`, `finance`.

### Проверь себя
- Есть ли оба набора данных?
- Есть ли колонки `model`, `feature_set`?

### Мини-вывод
Запишите 2-4 предложения: что получили в шаге, какой риск/компромисс увидели, что это меняет в следующем шаге.

Переход к следующему шагу: гипотезы зафиксированы, теперь считаем метрики калибровки на проверочной выборке `validation`.


In [ ]:
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets_raw()
hypotheses = lab.load_lab03_hypotheses()

print('Datasets:', sorted(datasets.keys()))
hypotheses


### TODO(обязательно): Мини-вывод по шагу 1
Объясните, почему гипотеза из ЛР 03 подходит как отправная точка для аудита калибровки.


## Шаг 2. Расчет метрик калибровки на проверочной выборке `validation`

### Что делаем
1. На мини-примере считаем Brier/LogLoss/ECE вручную через формулы.
2. Обучаем `uncalibrated`, `calibrated_sigmoid`, `calibrated_isotonic`.
3. Формируем `calibration_audit` только на `validation`.

### Зачем
Чтобы видеть, как калибровка влияет именно на качество вероятностей.

### Вход
`hypotheses`, наборы данных, наборы признаков.

### Выход
`calibration_audit` с колонками контракта.

### Проверь себя
- В `calibration_audit` только `split='validation'`?
- На каждом наборе данных ровно 3 варианта модели?

### Теория шага (интуиция + формулы)
Мы оцениваем качество вероятностей, а не только качество классов.

- **Brier score**: средний квадрат ошибки вероятности. Чем меньше, тем лучше.
  
  $\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2$
- **LogLoss**: штрафует уверенные ошибки сильнее.
  
  $\text{LogLoss} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log p_i + (1-y_i)\log(1-p_i)\right]$
- **ECE** (Expected Calibration Error): средний по бинам разрыв между confidence и фактической частотой.
  
  $\text{ECE} = \sum_{b=1}^{B}\frac{n_b}{N}\left|\text{conf}(b)-\text{acc}(b)\right|$

### Мини-вывод
Запишите 2-4 предложения: что получили в шаге, какой риск/компромисс увидели, что это меняет в следующем шаге.

Переход к следующему шагу: метрики посчитаны, теперь проверим графиками, где вероятности завышены или занижены.


In [ ]:
# Мини-числовой пример: считаем вероятностные метрики руками
mini_y = np.array([1, 0, 1, 0, 1, 0], dtype=int)
mini_p = np.array([0.90, 0.70, 0.40, 0.20, 0.60, 0.10], dtype=float)

mini_brier = float(np.mean((mini_p - mini_y) ** 2))
mini_logloss = float(
    -np.mean(mini_y * np.log(np.clip(mini_p, 1e-6, 1 - 1e-6)) + (1 - mini_y) * np.log(np.clip(1 - mini_p, 1e-6, 1 - 1e-6)))
)
mini_ece = float(lab.compute_ece(mini_y, mini_p, n_bins=3))

pd.DataFrame(
    {
        'metric': ['brier', 'log_loss', 'ece'],
        'value': [mini_brier, mini_logloss, mini_ece],
    }
)


In [ ]:
# Основной цикл: обучение вариантов модели и сбор аудита калибровки
calibration_records = []
trained_context = {}

for row in hypotheses.itertuples(index=False):
    dataset_name = row.dataset
    model_name = row.model
    feature_set_name = row.feature_set

    # 1) Берем данные и делим 60/20/20 (train/validation/test)
    df = datasets[dataset_name]
    x, y = lab.split_xy(df)
    x_train, x_valid, x_test, y_train, y_valid, y_test = lab.train_valid_test_split_stratified(x, y)

    # 2) Подготавливаем только признаки, выбранные в ЛР 03
    selected_features = lab.get_feature_set_features(feature_sets, dataset_name, feature_set_name)
    x_train_s, x_valid_s, x_test_s, selected_feature_names = lab.prepare_selected_matrices(
        x_train=x_train,
        x_valid=x_valid,
        x_test=x_test,
        selected_features=selected_features,
    )

    # TODO(обязательно): проверьте аргументы train_model_variants и поясните их смысл в markdown ниже
    variants = lab.train_model_variants(
        model_name=model_name,
        x_train=x_train_s,
        y_train=y_train,
    )

    trained_context[dataset_name] = {
        'model': model_name,
        'x_valid_selected': x_valid_s,
        'y_valid': y_valid,
        'variants': variants,
        'selected_feature_count': len(selected_feature_names),
    }

    # 3) Считаем метрики калибровки строго на проверочной выборке `validation`
    for variant_name, model in variants.items():
        valid_scores = lab.get_binary_score_vector(model, x_valid_s)
        calibration_records.append(
            lab.build_calibration_record(
                dataset_name=dataset_name,
                model_name=model_name,
                variant=variant_name,
                split='validation',
                y_true=y_valid,
                y_score=valid_scores,
            )
        )

calibration_audit = (
    pd.DataFrame(calibration_records)
    .loc[:, lab.CALIBRATION_AUDIT_COLUMNS]
    .sort_values(['dataset', 'variant', 'split'])
    .reset_index(drop=True)
)
calibration_audit


### TODO(обязательно): Мини-вывод по шагу 2
Сравните 3 варианта модели на каждом наборе данных по `brier`, `log_loss`, `ece` и объясните, какой вариант пока выглядит сильнее.


## Шаг 3. Анализ надежности вероятностей (reliability) и обязательные графики

### Что делаем
- Строим таблицу надежности (reliability) по бинам вероятностей.
- Визуализируем:
  1) диаграмму надежности (reliability diagram),
  2) средний разрыв калибровки (calibration gap),
  3) распределение вероятностей по классам.

### Зачем
Чтобы новички видели не только числа в CSV, но и визуальную картину пере/недокалибровки.

### Вход
`trained_context`, `calibration_audit`.

### Выход
`reliability_summary` + 3 графика.

### Проверь себя
- Есть ли диагональ идеальной калибровки на графике?
- Видно ли, какие варианты ближе к диагонали?

### Мини-вывод
Запишите 2-4 предложения: что получили в шаге, какой риск/компромисс увидели, что это меняет в следующем шаге.

Переход к следующему шагу: по графикам и метрикам выбираем лучший калиброванный вариант для каждого набора данных.


In [ ]:
def build_reliability_table(dataset_name, variant_name, y_true, y_prob, n_bins=10):
    y_true_arr = np.asarray(y_true, dtype=int)
    y_prob_arr = np.clip(np.asarray(y_prob, dtype=float), 0.0, 1.0)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.digitize(y_prob_arr, bins, right=True)

    rows = []
    for b in range(1, n_bins + 1):
        mask = bin_id == b
        if not np.any(mask):
            continue
        prob_mean = float(y_prob_arr[mask].mean())
        target_rate = float(y_true_arr[mask].mean())
        rows.append(
            {
                'dataset': dataset_name,
                'variant': variant_name,
                'bin': b,
                'n': int(mask.sum()),
                'prob_mean': prob_mean,
                'target_rate': target_rate,
                'abs_gap': abs(prob_mean - target_rate),
            }
        )
    return pd.DataFrame(rows)


reliability_frames = []
distribution_frames = []

for dataset_name, info in trained_context.items():
    y_valid = info['y_valid']
    for variant_name, model in info['variants'].items():
        valid_scores = lab.get_binary_score_vector(model, info['x_valid_selected'])
        reliability_frames.append(build_reliability_table(dataset_name, variant_name, y_valid, valid_scores, n_bins=10))

        dist_frame = pd.DataFrame(
            {
                'dataset': dataset_name,
                'variant': variant_name,
                'score': valid_scores,
                'target': y_valid.values,
            }
        )
        distribution_frames.append(dist_frame)

reliability_summary = pd.concat(reliability_frames, ignore_index=True)
score_distribution = pd.concat(distribution_frames, ignore_index=True)
reliability_summary.head(10)


In [ ]:
# График 1: Reliability diagram (по dataset, по variant)
datasets_sorted = sorted(reliability_summary['dataset'].unique())
fig, axes = plt.subplots(1, len(datasets_sorted), figsize=(7 * len(datasets_sorted), 5), squeeze=False)

for idx, dataset_name in enumerate(datasets_sorted):
    ax = axes[0, idx]
    ds = reliability_summary[reliability_summary['dataset'] == dataset_name]
    sns.lineplot(data=ds, x='prob_mean', y='target_rate', hue='variant', marker='o', ax=ax)
    ax.plot([0, 1], [0, 1], '--', color='black', label='ideal')
    ax.set_title(f'Reliability diagram: {dataset_name}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed positive rate')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


In [ ]:
# График 2: Средний разрыв калибровки (calibration gap)
gap_summary = (
    reliability_summary.groupby(['dataset', 'variant'], as_index=False)['abs_gap']
    .mean()
    .rename(columns={'abs_gap': 'mean_abs_gap'})
)

plt.figure(figsize=(9, 5))
sns.barplot(data=gap_summary, x='dataset', y='mean_abs_gap', hue='variant')
plt.title('Mean absolute calibration gap by dataset and variant')
plt.ylabel('Mean |prob_mean - target_rate|')
plt.xlabel('Dataset')
plt.tight_layout()
plt.show()

gap_summary


In [ ]:
# График 3: Распределение вероятностей по классам (validation)
fig, axes = plt.subplots(len(datasets_sorted), 1, figsize=(10, 5 * len(datasets_sorted)), squeeze=False)

for idx, dataset_name in enumerate(datasets_sorted):
    ax = axes[idx, 0]
    subset = score_distribution[score_distribution['dataset'] == dataset_name].copy()
    subset['target_label'] = subset['target'].map({0: 'class=0', 1: 'class=1'})
    sns.histplot(
        data=subset,
        x='score',
        hue='target_label',
        multiple='layer',
        stat='density',
        common_norm=False,
        bins=20,
        alpha=0.35,
        ax=ax,
    )
    ax.set_title(f'Probability distribution by class: {dataset_name}')
    ax.set_xlabel('Predicted probability')

plt.tight_layout()
plt.show()


### TODO(обязательно): Мини-вывод по шагу 3
По каждому набору данных объясните, какие визуальные признаки показали лучшую калибровку и почему.


## Шаг 4. Выбор `calibrated_best` по проверочной выборке `validation`

### Что делаем
Выбираем лучший калиброванный вариант на каждом наборе данных и сравниваем его с `uncalibrated`.

### Зачем
Нам нужен один калиброванный кандидат для следующего ноутбука с подбором порога.

### Вход
`calibration_audit` только по `validation`.

### Выход
`calibrated_best_summary`.

### Проверь себя
- Для каждого набора данных выбран ровно один `calibrated_best_source`?
- Выбор сделан только по проверочной выборке `validation`?

### Мини-вывод
Запишите 2-4 предложения: что получили в шаге, какой риск/компромисс увидели, что это меняет в следующем шаге.

Переход к следующему шагу: лучший вариант выбран, остается сохранить артефакт `calibration_audit.csv` для ноутбука 2.


In [ ]:
summary_rows = []
for dataset_name in sorted(lab.DATASET_PATHS):
    best_variant = lab.choose_best_calibrated_variant(calibration_audit, dataset_name=dataset_name)

    uncalibrated_row = calibration_audit[
        (calibration_audit['dataset'] == dataset_name)
        & (calibration_audit['variant'] == 'uncalibrated')
    ].iloc[0]
    best_row = calibration_audit[
        (calibration_audit['dataset'] == dataset_name)
        & (calibration_audit['variant'] == best_variant)
    ].iloc[0]

    summary_rows.append(
        {
            'dataset': dataset_name,
            'calibrated_best_source': best_variant,
            'uncalibrated_brier': float(uncalibrated_row['brier']),
            'best_brier': float(best_row['brier']),
            'uncalibrated_ece': float(uncalibrated_row['ece']),
            'best_ece': float(best_row['ece']),
        }
    )

calibrated_best_summary = pd.DataFrame(summary_rows)
calibrated_best_summary


In [ ]:
# Визуальное сравнение uncalibrated vs calibrated_best
compare_plot = calibrated_best_summary.melt(
    id_vars=['dataset', 'calibrated_best_source'],
    value_vars=['uncalibrated_brier', 'best_brier', 'uncalibrated_ece', 'best_ece'],
    var_name='metric_variant',
    value_name='value',
)

compare_plot['metric'] = compare_plot['metric_variant'].str.extract('(brier|ece)')
compare_plot['variant_group'] = compare_plot['metric_variant'].str.replace('_brier|_ece', '', regex=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=compare_plot, x='dataset', y='value', hue='metric_variant')
plt.title('Validation comparison: uncalibrated vs calibrated_best source')
plt.ylabel('Metric value (lower is better for Brier/ECE)')
plt.xlabel('Dataset')
plt.tight_layout()
plt.show()


### TODO(обязательно): Мини-вывод по шагу 4
Сформулируйте правило выбора `calibrated_best` своими словами и зафиксируйте, какой источник выбран для каждого набора данных.


## Шаг 5. Экспорт обязательного артефакта

### Что делаем
Экспортируем `calibration_audit.csv` в `outputs/`.

### Зачем
Этот файл — обязательный вход для ноутбука 2.

### Вход
`calibration_audit`.

### Выход
`outputs/calibration_audit.csv`.

### Проверь себя
- Колонки совпадают с контрактом?
- В `split` только `validation`?

### Мини-вывод
Запишите 2-4 предложения: что получили в шаге, какой риск/компромисс увидели, что это меняет в следующем шаге.

Переход к следующему шагу: этот ноутбук завершен, дальше в ноутбуке 2 будем выбирать порог и финальное правило решения.


In [ ]:
# TODO(обязательно):
# 1) Удалите намеренная остановка.
# 2) Сохраните calibration_audit в outputs/calibration_audit.csv.

raise NotImplementedError(
    'TODO(обязательно): сохраните calibration_audit.csv в outputs/ и удалите этот намеренная остановка.'
)
